# Distillation Reidentification Supervisor

Canonical unified distillation notebook for low-rank online reidentification with dual RL blending and optional observer refresh.

## User Config And Imports

In [ ]:
from pathlib import Path
import os

import numpy as np
import torch

from SACAgent.sac_agent import SACAgent
from Simulation.mpc import MpcSolverGeneral
from TD3Agent.agent import TD3Agent
from systems.distillation import DISTILLATION_SYSTEM_METADATA, get_distillation_notebook_defaults
from systems.distillation.data_io import canonical_baseline_path, load_distillation_system_data
from systems.distillation.plant import build_distillation_system, distillation_system_stepper
from systems.distillation.scenarios import build_distillation_disturbance_schedule
from utils.helpers import apply_min_max
from utils.notebook_setup import prepare_distillation_notebook_env, print_grouped_notebook_summary
from utils.plotting import compare_mpc_rl_from_dirs, plot_reidentification_results
from utils.reidentification import get_reidentification_state_dim
from utils.reidentification_runner import run_reidentification_supervisor
from utils.rewards import make_reward_fn_relative_QR

NB = get_distillation_notebook_defaults("reidentification")
AGENT_KIND = NB["agent_kind"]
RUN_MODE = NB["run_mode"]
DISTURBANCE_PROFILE = NB["disturbance_profile"]
STATE_MODE = NB["state_mode"]
STYLE_PROFILE = NB["style_profile"]
SAVE_PDF = NB["save_pdf"]
ASPEN_PRESET = NB["aspen_preset"]
ASPEN_PATH_OVERRIDE = NB["aspen_path_override"]
SNAPS_PATH_OVERRIDE = NB["snaps_path_override"]
ASPEN_ROOT_OVERRIDE = NB["aspen_root_override"]
DISTILLATION_VISIBLE = NB["distillation_visible"]
DISTILLATION_DATA_DIR_OVERRIDE = NB["data_dir_override"]
DISTILLATION_RESULTS_DIR_OVERRIDE = NB["results_dir_override"]
RESULT_PREFIX_OVERRIDE = NB["result_prefix_override"]
COMPARE_PREFIX_OVERRIDE = NB["compare_prefix_override"]
BASELINE_MPC_PATH_OVERRIDE = NB["baseline_mpc_path_override"]
N_TESTS_OVERRIDE = NB["n_tests_override"]
SET_POINTS_LEN_OVERRIDE = NB["set_points_len_override"]
WARM_START_OVERRIDE = NB["warm_start_override"]
TEST_CYCLE_OVERRIDE = NB["test_cycle_override"]
PLOT_START_EPISODE_OVERRIDE = NB["plot_start_episode_override"]
COMPARE_START_EPISODE_OVERRIDE = NB["compare_start_episode_override"]
REPO_ROOT, DATA_DIR, RESULT_DIR, DISTURBANCE_PROFILE, DYN_PATH, SNAPS_PATH, ASPEN_SOURCE = prepare_distillation_notebook_env(run_mode=RUN_MODE, disturbance_profile=DISTURBANCE_PROFILE, family="reidentification", aspen_preset=ASPEN_PRESET, dyn_path_override=ASPEN_PATH_OVERRIDE, snaps_path_override=SNAPS_PATH_OVERRIDE, aspen_root_override=ASPEN_ROOT_OVERRIDE, data_dir_override=DISTILLATION_DATA_DIR_OVERRIDE, results_dir_override=DISTILLATION_RESULTS_DIR_OVERRIDE)
os.chdir(REPO_ROOT)


## System And Data Setup

In [ ]:
SYS = NB["system_setup"]
RUN_PROFILE = NB["run_profiles"][(AGENT_KIND, RUN_MODE, DISTURBANCE_PROFILE)]
nominal_conditions = SYS["nominal_conditions"].copy()
ss_inputs = SYS["ss_inputs"].copy()
u_min = SYS["input_bounds"]["u_min"].copy()
u_max = SYS["input_bounds"]["u_max"].copy()
setpoint_y = SYS["setpoint_range_phys"].copy()
y_sp_scenario_phys = SYS["rl_setpoints_phys"].copy()
system = build_distillation_system(path=DYN_PATH, ss_inputs=ss_inputs, initialization_point=nominal_conditions, delta_t=SYS["delta_t_hours"], visible=DISTILLATION_VISIBLE)
steady_states = {"ss_inputs": np.asarray(system.ss_inputs, float).copy(), "y_ss": np.asarray(system.y_ss, float).copy()}
system_data = load_distillation_system_data(REPO_ROOT, steady_states, setpoint_y, u_min, u_max, data_override=DISTILLATION_DATA_DIR_OVERRIDE)
A_phys = np.asarray(system_data["A"], float)
B_phys = np.asarray(system_data["B"], float)
C_phys = np.asarray(system_data["C"], float)
A_aug = np.asarray(system_data["A_aug"], float)
B_aug = np.asarray(system_data["B_aug"], float)
C_aug = np.asarray(system_data["C_aug"], float)
data_min = np.asarray(system_data["data_min"], float)
data_max = np.asarray(system_data["data_max"], float)
min_max_dict = system_data["min_max_dict"]
inputs_number = int(B_aug.shape[1])
y_sp_scenario = apply_min_max(y_sp_scenario_phys, data_min[inputs_number:], data_max[inputs_number:]) - apply_min_max(steady_states["y_ss"], data_min[inputs_number:], data_max[inputs_number:])
RESULT_PREFIX = RESULT_PREFIX_OVERRIDE or f"distillation_reidentification_{AGENT_KIND}_{RUN_MODE}_{DISTURBANCE_PROFILE}_{STATE_MODE}_unified"
COMPARE_PREFIX = COMPARE_PREFIX_OVERRIDE or f"distillation_compare_reidentification_{AGENT_KIND}_{RUN_MODE}_{DISTURBANCE_PROFILE}_{STATE_MODE}"
BASELINE_MPC_PATH = Path(BASELINE_MPC_PATH_OVERRIDE).expanduser() if BASELINE_MPC_PATH_OVERRIDE else canonical_baseline_path(REPO_ROOT, RUN_MODE, DISTURBANCE_PROFILE, data_override=DISTILLATION_DATA_DIR_OVERRIDE)
n_tests = int(RUN_PROFILE["n_tests"] if N_TESTS_OVERRIDE is None else N_TESTS_OVERRIDE)
set_points_len = int(RUN_PROFILE["set_points_len"] if SET_POINTS_LEN_OVERRIDE is None else SET_POINTS_LEN_OVERRIDE)
warm_start = int(RUN_PROFILE["warm_start"] if WARM_START_OVERRIDE is None else WARM_START_OVERRIDE)
TEST_CYCLE = list(RUN_PROFILE["test_cycle"] if TEST_CYCLE_OVERRIDE is None else TEST_CYCLE_OVERRIDE)
PLOT_START_EPISODE = int(RUN_PROFILE.get("plot_start_episode", 1) if PLOT_START_EPISODE_OVERRIDE is None else PLOT_START_EPISODE_OVERRIDE)
COMPARE_START_EPISODE = int(RUN_PROFILE.get("compare_start_episode", PLOT_START_EPISODE) if COMPARE_START_EPISODE_OVERRIDE is None else COMPARE_START_EPISODE_OVERRIDE)
TOTAL_STEPS = n_tests * set_points_len * len(y_sp_scenario_phys)
DISTURBANCE_NOMINAL_FEED = float(system.feed.FmR.Value)
DISTURBANCE_SCHEDULE = build_distillation_disturbance_schedule(RUN_MODE, DISTURBANCE_PROFILE, TOTAL_STEPS, nominal_feed=DISTURBANCE_NOMINAL_FEED)


## Run / Reward / Agent Setup

In [ ]:
CTRL = NB["controller"]
REID = NB["reidentification"]
TD3_CFG = NB["td3_agent"]
SAC_CFG = NB["sac_agent"]
REWARD_CFG = NB["reward"]
poles = SYS["observer_poles"].copy()
N_INPUTS = int(B_aug.shape[1])
N_OUTPUTS = int(C_aug.shape[0])
STATE_DIM = get_reidentification_state_dim(A_aug.shape[0], N_OUTPUTS, N_INPUTS, STATE_MODE)
MISMATCH_CLIP = CTRL["mismatch_clip"]
INNOVATION_SCALE_MODE = CTRL["innovation_scale_mode"]
INNOVATION_SCALE_REF = CTRL["innovation_scale_ref"]
TRACKING_SCALE_MODE = CTRL["tracking_scale_mode"]
TRACKING_ETA_TOL = CTRL["tracking_eta_tol"]
TRACKING_SCALE_FLOOR = CTRL["tracking_scale_floor"]
TRACKING_SCALE_FLOOR_MODE = CTRL["tracking_scale_floor_mode"]
BASE_STATE_NORM_MODE = CTRL["base_state_norm_mode"]
BASE_STATE_RUNNING_NORM_CLIP = CTRL["base_state_running_norm_clip"]
BASE_STATE_RUNNING_NORM_EPS = CTRL["base_state_running_norm_eps"]
MISMATCH_FEATURE_TRANSFORM_MODE = CTRL["mismatch_feature_transform_mode"]
MISMATCH_TRANSFORM_TANH_SCALE = CTRL["mismatch_transform_tanh_scale"]
MISMATCH_TRANSFORM_POST_CLIP = CTRL["mismatch_transform_post_clip"]
ACTION_DIM = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
predict_h = CTRL["predict_h"]
cont_h = CTRL["cont_h"]
Q1_penalty = CTRL["Q1_penalty"]
Q2_penalty = CTRL["Q2_penalty"]
R1_penalty = CTRL["R1_penalty"]
R2_penalty = CTRL["R2_penalty"]
USE_SHIFTED_MPC_WARM_START = CTRL["use_shifted_mpc_warm_start"]
BASIS_FAMILY = REID["basis_family"]
ID_COMPONENT_MODE = REID["id_component_mode"]
OBSERVER_UPDATE_ALIGNMENT = REID["observer_update_alignment"]
CANDIDATE_GUARD_MODE = REID["candidate_guard_mode"]
NORMALIZE_BLEND_EXTRAS = bool(REID["normalize_blend_extras"])
BLEND_EXTRA_CLIP = float(REID["blend_extra_clip"])
BLEND_RESIDUAL_SCALE = float(REID["blend_residual_scale"])
LOG_THETA_CLIPPING = bool(REID["log_theta_clipping"])
ID_SOLVER = REID["id_solver"]
RANK_A = int(REID["rank_A"])
RANK_B = int(REID["rank_B"])
OFFLINE_WINDOW = int(REID["offline_window"])
OFFLINE_STRIDE = int(REID["offline_stride"])
LAMBDA_A_OFF = float(REID["lambda_A_off"])
LAMBDA_B_OFF = float(REID["lambda_B_off"])
ID_WINDOW = int(REID["id_window"])
ID_UPDATE_PERIOD = int(REID["id_update_period"])
LAMBDA_PREV_A = float(REID["lambda_prev_A"])
LAMBDA_PREV_B = float(REID["lambda_prev_B"])
LAMBDA_0_A = float(REID["lambda_0_A"])
LAMBDA_0_B = float(REID["lambda_0_B"])
THETA_LOW_A = float(REID["theta_low_A"])
THETA_HIGH_A = float(REID["theta_high_A"])
THETA_LOW_B = float(REID["theta_low_B"])
THETA_HIGH_B = float(REID["theta_high_B"])
DELTA_A_MAX = float(REID["delta_A_max"])
DELTA_B_MAX = float(REID["delta_B_max"])
ETA_TAU_A = float(REID["eta_tau_A"])
ETA_TAU_B = float(REID["eta_tau_B"])
FREEZE_IDENTIFICATION_DURING_WARM_START = bool(REID["freeze_identification_during_warm_start"])
GUARD_VALIDATION_FRACTION = float(REID["guard_validation_fraction"])
GUARD_MIN_VALIDATION_SAMPLES = int(REID["guard_min_validation_samples"])
GUARD_MIN_TRAIN_SAMPLES = int(REID["guard_min_train_samples"])
MAX_THETA_CLIPPED_FRACTION = float(REID["max_theta_clipped_fraction"])
MAX_CONDITION_NUMBER = float(REID["max_condition_number"])
MAX_VALIDATION_RESIDUAL_RATIO = float(REID["max_validation_residual_ratio"])
MAX_FULL_RESIDUAL_RATIO = float(REID["max_full_residual_ratio"])
BLEND_VALIDITY_MODE = REID["blend_validity_mode"]
BLEND_VALIDITY_SCALE_FLOOR = float(REID["blend_validity_scale_floor"])
BLEND_VALIDITY_RESIDUAL_SOFT = float(REID["blend_validity_residual_soft"])
BLEND_VALIDITY_RESIDUAL_HARD = float(REID["blend_validity_residual_hard"])
BLEND_VALIDITY_CLIPPED_SOFT = float(REID["blend_validity_clipped_soft"])
BLEND_VALIDITY_CLIPPED_HARD = float(REID["blend_validity_clipped_hard"])
BLEND_VALIDITY_CONDITION_SOFT = float(REID["blend_validity_condition_soft"])
BLEND_VALIDITY_CONDITION_HARD = float(REID["blend_validity_condition_hard"])
BLEND_VALIDITY_FALLBACK_SCALE = float(REID["blend_validity_fallback_scale"])
BLEND_VALIDITY_INVALID_CANDIDATE_SCALE = float(REID["blend_validity_invalid_candidate_scale"])
OBSERVER_REFRESH_ENABLED = bool(REID["observer_refresh_enabled"])
OBSERVER_REFRESH_EVERY_EPISODES = int(REID["observer_refresh_every_episodes"])
RHO_OBS = float(REID["rho_obs"])
FORCE_ETA_CONSTANT = REID["force_eta_constant"]
DISABLE_IDENTIFICATION = bool(REID["disable_identification"])
TD3_EXPLORATION_MODE = TD3_CFG["exploration_mode"]
TD3_N_STEP = int(TD3_CFG["n_step"])
TD3_MULTISTEP_MODE = TD3_CFG["multistep_mode"]
TD3_LAMBDA_VALUE = float(TD3_CFG["lambda_value"])
TD3_LOSS_TYPE = TD3_CFG["loss_type"]
TD3_PARAM_NOISE_RESAMPLE_INTERVAL = TD3_CFG["param_noise_resample_interval"]
TD3_BUFFER_SIZE = int(TD3_CFG["buffer_size"])
TD3_REPLAY_FRAC_PER = float(TD3_CFG["replay_frac_per"])
TD3_REPLAY_FRAC_RECENT = float(TD3_CFG["replay_frac_recent"])
TD3_REPLAY_RECENT_WINDOW_MULT = int(TD3_CFG["replay_recent_window_mult"])
TD3_REPLAY_RECENT_WINDOW = int(TD3_CFG["replay_recent_window"]) if TD3_CFG["replay_recent_window"] is not None else min(TD3_BUFFER_SIZE, TD3_REPLAY_RECENT_WINDOW_MULT * set_points_len)
TD3_REPLAY_ALPHA = float(TD3_CFG["replay_alpha"])
TD3_REPLAY_BETA_START = float(TD3_CFG["replay_beta_start"])
TD3_REPLAY_BETA_END = float(TD3_CFG["replay_beta_end"])
TD3_REPLAY_BETA_STEPS = int(TD3_CFG["replay_beta_steps"])
SAC_BUFFER_SIZE = int(SAC_CFG["buffer_size"])
SAC_REPLAY_FRAC_PER = float(SAC_CFG["replay_frac_per"])
SAC_REPLAY_FRAC_RECENT = float(SAC_CFG["replay_frac_recent"])
SAC_REPLAY_RECENT_WINDOW_MULT = int(SAC_CFG["replay_recent_window_mult"])
SAC_REPLAY_RECENT_WINDOW = int(SAC_CFG["replay_recent_window"]) if SAC_CFG["replay_recent_window"] is not None else min(SAC_BUFFER_SIZE, SAC_REPLAY_RECENT_WINDOW_MULT * set_points_len)
SAC_REPLAY_ALPHA = float(SAC_CFG["replay_alpha"])
SAC_REPLAY_BETA_START = float(SAC_CFG["replay_beta_start"])
SAC_REPLAY_BETA_END = float(SAC_CFG["replay_beta_end"])
SAC_REPLAY_BETA_STEPS = int(SAC_CFG["replay_beta_steps"])
SAC_LOSS_TYPE = SAC_CFG["loss_type"]
SAC_N_STEP = int(SAC_CFG["n_step"])
SAC_MULTISTEP_MODE = SAC_CFG["multistep_mode"]
SAC_LAMBDA_VALUE = float(SAC_CFG["lambda_value"])
reward_params, reward_fn = make_reward_fn_relative_QR(data_min, data_max, N_INPUTS, **REWARD_CFG)
MPC_obj = MpcSolverGeneral(A_aug, B_aug, C_aug, Q_out=np.array([Q1_penalty, Q2_penalty], float), R_in=np.array([R1_penalty, R2_penalty], float), NP=predict_h, NC=cont_h)
if AGENT_KIND == "td3":
    reid_agent = TD3Agent(state_dim=STATE_DIM, action_dim=ACTION_DIM, actor_hidden=list(TD3_CFG["actor_hidden"]), critic_hidden=list(TD3_CFG["critic_hidden"]), gamma=TD3_CFG["gamma"], actor_lr=TD3_CFG["actor_lr"], critic_lr=TD3_CFG["critic_lr"], batch_size=TD3_CFG["batch_size"], policy_delay=TD3_CFG["policy_delay"], target_policy_smoothing_noise_std=TD3_CFG["target_policy_smoothing_noise_std"], noise_clip=TD3_CFG["noise_clip"], max_action=TD3_CFG["max_action"], tau=TD3_CFG["tau"], std_start=TD3_CFG["std_start"], std_end=TD3_CFG["std_end"], std_decay_rate=TD3_CFG["std_decay_rate"], std_decay_mode=TD3_CFG["std_decay_mode"], buffer_size=TD3_BUFFER_SIZE, replay_frac_per=TD3_REPLAY_FRAC_PER, replay_frac_recent=TD3_REPLAY_FRAC_RECENT, replay_recent_window=TD3_REPLAY_RECENT_WINDOW, replay_alpha=TD3_REPLAY_ALPHA, replay_beta_start=TD3_REPLAY_BETA_START, replay_beta_end=TD3_REPLAY_BETA_END, replay_beta_steps=TD3_REPLAY_BETA_STEPS, device=DEVICE, actor_freeze=TD3_CFG["actor_freeze"], exploration_mode=TD3_EXPLORATION_MODE, loss_type=TD3_LOSS_TYPE, param_noise_resample_interval=TD3_PARAM_NOISE_RESAMPLE_INTERVAL, n_step=TD3_N_STEP, multistep_mode=TD3_MULTISTEP_MODE, lambda_value=TD3_LAMBDA_VALUE)
else:
    target_entropy = -ACTION_DIM if SAC_CFG["target_entropy"] == "auto_negative_action_dim" else SAC_CFG["target_entropy"]
    reid_agent = SACAgent(state_dim=STATE_DIM, action_dim=ACTION_DIM, actor_hidden=list(SAC_CFG["actor_hidden"]), critic_hidden=list(SAC_CFG["critic_hidden"]), gamma=SAC_CFG["gamma"], actor_lr=SAC_CFG["actor_lr"], critic_lr=SAC_CFG["critic_lr"], alpha_lr=SAC_CFG["alpha_lr"], batch_size=SAC_CFG["batch_size"], grad_clip_norm=SAC_CFG["grad_clip_norm"], init_alpha=SAC_CFG["init_alpha"], learn_alpha=SAC_CFG["learn_alpha"], target_entropy=target_entropy, target_update=SAC_CFG["target_update"], tau=SAC_CFG["tau"], hard_update_interval=SAC_CFG["hard_update_interval"], activation=SAC_CFG["activation"], use_layernorm=SAC_CFG["use_layernorm"], dropout=SAC_CFG["dropout"], max_action=SAC_CFG["max_action"], buffer_size=SAC_BUFFER_SIZE, replay_frac_per=SAC_REPLAY_FRAC_PER, replay_frac_recent=SAC_REPLAY_FRAC_RECENT, replay_recent_window=SAC_REPLAY_RECENT_WINDOW, replay_alpha=SAC_REPLAY_ALPHA, replay_beta_start=SAC_REPLAY_BETA_START, replay_beta_end=SAC_REPLAY_BETA_END, replay_beta_steps=SAC_REPLAY_BETA_STEPS, device=DEVICE, use_adamw=SAC_CFG["use_adamw"], actor_freeze=SAC_CFG["actor_freeze"], loss_type=SAC_LOSS_TYPE, n_step=SAC_N_STEP, multistep_mode=SAC_MULTISTEP_MODE, lambda_value=SAC_LAMBDA_VALUE)
REPLAY_SETTINGS = {
    "buffer_size": TD3_BUFFER_SIZE if AGENT_KIND == "td3" else SAC_BUFFER_SIZE,
    "replay_frac_per": TD3_REPLAY_FRAC_PER if AGENT_KIND == "td3" else SAC_REPLAY_FRAC_PER,
    "replay_frac_recent": TD3_REPLAY_FRAC_RECENT if AGENT_KIND == "td3" else SAC_REPLAY_FRAC_RECENT,
    "replay_recent_window": TD3_REPLAY_RECENT_WINDOW if AGENT_KIND == "td3" else SAC_REPLAY_RECENT_WINDOW,
    "replay_alpha": TD3_REPLAY_ALPHA if AGENT_KIND == "td3" else SAC_REPLAY_ALPHA,
    "replay_beta_start": TD3_REPLAY_BETA_START if AGENT_KIND == "td3" else SAC_REPLAY_BETA_START,
    "replay_beta_end": TD3_REPLAY_BETA_END if AGENT_KIND == "td3" else SAC_REPLAY_BETA_END,
    "replay_beta_steps": TD3_REPLAY_BETA_STEPS if AGENT_KIND == "td3" else SAC_REPLAY_BETA_STEPS,
}


## Resolved Summary

In [ ]:
print_grouped_notebook_summary(
    "Resolved Distillation Reidentification Run",
    {
        "Paths": {"Repo root": REPO_ROOT, "Data dir": DATA_DIR, "Results dir": RESULT_DIR, "Aspen source": ASPEN_SOURCE, "Dyn path": DYN_PATH, "Snaps path": SNAPS_PATH, "Baseline MPC": BASELINE_MPC_PATH},
        "Run setup": {"Agent kind": AGENT_KIND, "Run mode": RUN_MODE, "Disturbance profile": DISTURBANCE_PROFILE, "State mode": STATE_MODE, "n_tests": n_tests, "set_points_len": set_points_len, "warm_start": warm_start, "test_cycle": TEST_CYCLE, "use_shifted_mpc_warm_start": USE_SHIFTED_MPC_WARM_START},
        "System / controller": {"delta_t_hours": SYS["delta_t_hours"], "predict_h": predict_h, "cont_h": cont_h, "Q penalties": [Q1_penalty, Q2_penalty], "R penalties": [R1_penalty, R2_penalty], "observer_poles": poles.tolist()},
        "Identification": {"basis_family": BASIS_FAMILY, "id_component_mode": ID_COMPONENT_MODE, "candidate_guard_mode": CANDIDATE_GUARD_MODE, "observer_update_alignment": OBSERVER_UPDATE_ALIGNMENT, "normalize_blend_extras": NORMALIZE_BLEND_EXTRAS, "blend_extra_clip": BLEND_EXTRA_CLIP, "blend_residual_scale": BLEND_RESIDUAL_SCALE, "log_theta_clipping": LOG_THETA_CLIPPING, "id_solver": ID_SOLVER, "rank_A": RANK_A, "rank_B": RANK_B, "offline_window": OFFLINE_WINDOW, "offline_stride": OFFLINE_STRIDE, "lambda_A_off": LAMBDA_A_OFF, "lambda_B_off": LAMBDA_B_OFF, "id_window": ID_WINDOW, "id_update_period": ID_UPDATE_PERIOD, "lambda_prev_A": LAMBDA_PREV_A, "lambda_prev_B": LAMBDA_PREV_B, "lambda_0_A": LAMBDA_0_A, "lambda_0_B": LAMBDA_0_B, "theta_low_A": THETA_LOW_A, "theta_high_A": THETA_HIGH_A, "theta_low_B": THETA_LOW_B, "theta_high_B": THETA_HIGH_B, "delta_A_max": DELTA_A_MAX, "delta_B_max": DELTA_B_MAX, "eta_tau_A": ETA_TAU_A, "eta_tau_B": ETA_TAU_B, "freeze_id_during_warm_start": FREEZE_IDENTIFICATION_DURING_WARM_START, "guard_validation_fraction": GUARD_VALIDATION_FRACTION, "guard_min_validation_samples": GUARD_MIN_VALIDATION_SAMPLES, "guard_min_train_samples": GUARD_MIN_TRAIN_SAMPLES, "max_theta_clipped_fraction": MAX_THETA_CLIPPED_FRACTION, "max_condition_number": MAX_CONDITION_NUMBER, "max_validation_residual_ratio": MAX_VALIDATION_RESIDUAL_RATIO, "max_full_residual_ratio": MAX_FULL_RESIDUAL_RATIO, "blend_validity_mode": BLEND_VALIDITY_MODE, "blend_validity_residual_soft_hard": [BLEND_VALIDITY_RESIDUAL_SOFT, BLEND_VALIDITY_RESIDUAL_HARD], "blend_validity_clipped_soft_hard": [BLEND_VALIDITY_CLIPPED_SOFT, BLEND_VALIDITY_CLIPPED_HARD], "blend_validity_condition_soft_hard": [BLEND_VALIDITY_CONDITION_SOFT, BLEND_VALIDITY_CONDITION_HARD], "blend_validity_fallback_scale": BLEND_VALIDITY_FALLBACK_SCALE, "blend_validity_invalid_candidate_scale": BLEND_VALIDITY_INVALID_CANDIDATE_SCALE, "observer_refresh_enabled": OBSERVER_REFRESH_ENABLED, "observer_refresh_every_episodes": OBSERVER_REFRESH_EVERY_EPISODES, "rho_obs": RHO_OBS},
        "Agent": {"supervisor": "low-rank online re-identification + dual RL blend", "action_dim": ACTION_DIM},
        "Replay": REPLAY_SETTINGS,
        "Mismatch": {"clip": MISMATCH_CLIP, "innovation_scale_mode": INNOVATION_SCALE_MODE, "tracking_scale_mode": TRACKING_SCALE_MODE, "tracking_eta_tol": TRACKING_ETA_TOL, "tracking_scale_floor_mode": TRACKING_SCALE_FLOOR_MODE},
        "Plotting / export": {"style_profile": STYLE_PROFILE, "save_pdf": SAVE_PDF, "result_prefix": RESULT_PREFIX, "compare_prefix": COMPARE_PREFIX, "plot_start_episode": PLOT_START_EPISODE, "compare_start_episode": COMPARE_START_EPISODE},
    },
)


## Run

In [ ]:
reid_cfg = {
    "system_name": "distillation",
    "agent_kind": AGENT_KIND,
    "run_mode": RUN_MODE,
    "disturbance_profile": DISTURBANCE_PROFILE,
    "state_mode": STATE_MODE,
    "mismatch_clip": MISMATCH_CLIP,
    "innovation_scale_mode": INNOVATION_SCALE_MODE,
    "innovation_scale_ref": INNOVATION_SCALE_REF,
    "tracking_scale_mode": TRACKING_SCALE_MODE,
    "tracking_eta_tol": TRACKING_ETA_TOL,
    "tracking_scale_floor": TRACKING_SCALE_FLOOR,
    "tracking_scale_floor_mode": TRACKING_SCALE_FLOOR_MODE,
    "base_state_norm_mode": BASE_STATE_NORM_MODE,
    "base_state_running_norm_clip": BASE_STATE_RUNNING_NORM_CLIP,
    "base_state_running_norm_eps": BASE_STATE_RUNNING_NORM_EPS,
    "mismatch_feature_transform_mode": MISMATCH_FEATURE_TRANSFORM_MODE,
    "mismatch_transform_tanh_scale": MISMATCH_TRANSFORM_TANH_SCALE,
    "mismatch_transform_post_clip": MISMATCH_TRANSFORM_POST_CLIP,
    "notebook_source": "distillation_RL_assisted_MPC_reidentification_unified.ipynb",
    "n_tests": n_tests,
    "set_points_len": set_points_len,
    "warm_start": warm_start,
    "test_cycle": TEST_CYCLE,
    "predict_h": predict_h,
    "cont_h": cont_h,
    "use_shifted_mpc_warm_start": USE_SHIFTED_MPC_WARM_START,
    "basis_family": BASIS_FAMILY,
    "id_component_mode": ID_COMPONENT_MODE,
    "candidate_guard_mode": CANDIDATE_GUARD_MODE,
    "observer_update_alignment": OBSERVER_UPDATE_ALIGNMENT,
    "normalize_blend_extras": NORMALIZE_BLEND_EXTRAS,
    "blend_extra_clip": BLEND_EXTRA_CLIP,
    "blend_residual_scale": BLEND_RESIDUAL_SCALE,
    "log_theta_clipping": LOG_THETA_CLIPPING,
    "action_dim": ACTION_DIM,
    "id_solver": ID_SOLVER,
    "rank_A": RANK_A,
    "rank_B": RANK_B,
    "offline_window": OFFLINE_WINDOW,
    "offline_stride": OFFLINE_STRIDE,
    "lambda_A_off": LAMBDA_A_OFF,
    "lambda_B_off": LAMBDA_B_OFF,
    "id_window": ID_WINDOW,
    "id_update_period": ID_UPDATE_PERIOD,
    "lambda_prev_A": LAMBDA_PREV_A,
    "lambda_prev_B": LAMBDA_PREV_B,
    "lambda_0_A": LAMBDA_0_A,
    "lambda_0_B": LAMBDA_0_B,
    "theta_low_A": THETA_LOW_A,
    "theta_high_A": THETA_HIGH_A,
    "theta_low_B": THETA_LOW_B,
    "theta_high_B": THETA_HIGH_B,
    "delta_A_max": DELTA_A_MAX,
    "delta_B_max": DELTA_B_MAX,
    "eta_tau_A": ETA_TAU_A,
    "eta_tau_B": ETA_TAU_B,
    "freeze_identification_during_warm_start": FREEZE_IDENTIFICATION_DURING_WARM_START,
    "guard_validation_fraction": GUARD_VALIDATION_FRACTION,
    "guard_min_validation_samples": GUARD_MIN_VALIDATION_SAMPLES,
    "guard_min_train_samples": GUARD_MIN_TRAIN_SAMPLES,
    "max_theta_clipped_fraction": MAX_THETA_CLIPPED_FRACTION,
    "max_condition_number": MAX_CONDITION_NUMBER,
    "max_validation_residual_ratio": MAX_VALIDATION_RESIDUAL_RATIO,
    "max_full_residual_ratio": MAX_FULL_RESIDUAL_RATIO,
    "blend_validity_mode": BLEND_VALIDITY_MODE,
    "blend_validity_scale_floor": BLEND_VALIDITY_SCALE_FLOOR,
    "blend_validity_residual_soft": BLEND_VALIDITY_RESIDUAL_SOFT,
    "blend_validity_residual_hard": BLEND_VALIDITY_RESIDUAL_HARD,
    "blend_validity_clipped_soft": BLEND_VALIDITY_CLIPPED_SOFT,
    "blend_validity_clipped_hard": BLEND_VALIDITY_CLIPPED_HARD,
    "blend_validity_condition_soft": BLEND_VALIDITY_CONDITION_SOFT,
    "blend_validity_condition_hard": BLEND_VALIDITY_CONDITION_HARD,
    "blend_validity_fallback_scale": BLEND_VALIDITY_FALLBACK_SCALE,
    "blend_validity_invalid_candidate_scale": BLEND_VALIDITY_INVALID_CANDIDATE_SCALE,
    "observer_refresh_enabled": OBSERVER_REFRESH_ENABLED,
    "observer_refresh_every_episodes": OBSERVER_REFRESH_EVERY_EPISODES,
    "rho_obs": RHO_OBS,
    "force_eta_constant": FORCE_ETA_CONSTANT,
    "disable_identification": DISABLE_IDENTIFICATION,
    "nominal_qi": CTRL["nominal_qi"],
    "nominal_qs": CTRL["nominal_qs"],
    "nominal_ha": CTRL["nominal_ha"],
    "qi_change": CTRL["qi_change"],
    "qs_change": CTRL["qs_change"],
    "ha_change": CTRL["ha_change"],
    "Q1_penalty": Q1_penalty,
    "Q2_penalty": Q2_penalty,
    "R1_penalty": R1_penalty,
    "R2_penalty": R2_penalty,
    "b_min": system_data["b_min"],
    "b_max": system_data["b_max"],
    "repo_root": REPO_ROOT,
    "data_dir_override": DISTILLATION_DATA_DIR_OVERRIDE,
    "baseline_mpc_path": BASELINE_MPC_PATH,
}
runtime_ctx = {
    "system": system,
    "agent": reid_agent,
    "MPC_obj": MPC_obj,
    "steady_states": steady_states,
    "min_max_dict": min_max_dict,
    "data_min": data_min,
    "data_max": data_max,
    "repo_root": REPO_ROOT,
    "data_dir": DATA_DIR,
    "distillation_data_dir": DATA_DIR,
    "baseline_mpc_path": BASELINE_MPC_PATH,
    "A": A_phys,
    "B": B_phys,
    "C": C_phys,
    "A_aug": A_aug,
    "B_aug": B_aug,
    "C_aug": C_aug,
    "poles": poles,
    "y_sp_scenario": y_sp_scenario,
    "reward_fn": reward_fn,
    "system_metadata": DISTILLATION_SYSTEM_METADATA,
    "reward_params": reward_params,
    "system_stepper": distillation_system_stepper,
    "disturbance_labels": DISTILLATION_SYSTEM_METADATA.get("disturbance_labels"),
    "disturbance_schedule": DISTURBANCE_SCHEDULE,
    "disturbance_profile_name": DISTURBANCE_PROFILE,
}
result_bundle = run_reidentification_supervisor(reid_cfg=reid_cfg, runtime_ctx=runtime_ctx)
result_bundle["mpc_path_or_dir"] = BASELINE_MPC_PATH


## Plotting And Export

In [ ]:
# Generate saved figures and any requested MPC comparison plots.
out_dir_rl = plot_reidentification_results(
    result_bundle=result_bundle,
    plot_cfg={
        "directory": RESULT_DIR,
        "prefix_name": RESULT_PREFIX,
        "start_episode": PLOT_START_EPISODE,
        "save_pdf": SAVE_PDF,
        "style_profile": STYLE_PROFILE,
        "debug_id_plots": False,
    },
)

out_dir_cmp = compare_mpc_rl_from_dirs(
    rl_dir=out_dir_rl,
    mpc_path_or_dir=BASELINE_MPC_PATH,
    reward_fn=reward_fn,
    directory=RESULT_DIR,
    prefix_name=COMPARE_PREFIX,
    compare_mode=RUN_MODE,
    start_episode=COMPARE_START_EPISODE,
    save_pdf=SAVE_PDF,
    style_profile=STYLE_PROFILE,
)

print(out_dir_rl)
print(out_dir_cmp)

try:
    system.close(SNAPS_PATH)
except Exception:
    pass
